#### Loading and Transforming the tables


#### Calendar Table

In [0]:
# The Date Table

from pyspark.sql import functions as F

paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Calendar"

df_date = (
    spark.read
    .format("parquet")
    .load(paths)
)

df_date = df_date.withColumn("Date", F.to_date(F.col("Date"), 'M/d/yyyy'))
df_date = df_date.withColumn("Years", F.year(F.col("Date")))
df_date = df_date.withColumn("Months", F.month(F.col("Date")))
display(df_date.head(5))

### Load the transformed date table to silver layer
(
    df_date.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverCalenda.delta")
    .save()
)


### Load the transformed date table to gold layer
(
    df_date.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldCalenda.delta")
    .save()
)

Date,Years,Months
2015-01-01,2015,1
2015-01-02,2015,1
2015-01-03,2015,1
2015-01-04,2015,1
2015-01-05,2015,1


#### Customer Table

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Customers"

df_Customers = (
    spark.read
    .format("parquet")
    .load(paths)
)

df_Customers = (
    df_Customers
    .withColumn("FullName", F.concat_ws(" ", F.col("FirstName"), F.col("LastName")))
    .withColumn("AnnualIncome", F.col("AnnualIncome").cast("integer")).drop("Prefix", "FirstName", "LastName", "EmailAddress")
)

### Load the transformed customers table to silver layer
(
    df_Customers.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverCustomers.delta")
    .save()
)


### Load the transformed customers table to gold layer
(
    df_Customers.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldCustomers.delta")
    .save()
)
display(df_Customers.head(10))


CustomerKey,BirthDate,MaritalStatus,Gender,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,FullName
11000,1966-04-08,M,M,90000,2,Bachelors,Professional,true,JON YANG
11001,1965-05-14,S,M,60000,3,Bachelors,Professional,false,EUGENE HUANG
11002,1965-08-12,M,M,60000,3,Bachelors,Professional,true,RUBEN TORRES
11003,1968-02-15,S,F,70000,0,Bachelors,Professional,false,CHRISTY ZHU
11004,1968-08-08,S,F,80000,5,Bachelors,Professional,true,ELIZABETH JOHNSON
11005,1965-08-05,S,M,70000,0,Bachelors,Professional,true,JULIO RUIZ
11007,1964-05-09,M,M,60000,3,Bachelors,Professional,true,MARCO MEHTA
11008,1964-07-07,S,F,60000,4,Bachelors,Professional,true,ROBIN VERHOFF
11009,1964-04-01,S,M,70000,0,Bachelors,Professional,false,SHANNON CARLSON
11010,1964-02-06,S,F,70000,0,Bachelors,Professional,false,JACQUELYN SUAREZ


#### Product_Categories Table (No transformation)

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Product_Categories"

df_Product_Categories = (
    spark.read
    .format("parquet")
    .load(paths)
)

### Load the transformed Product_Categories table to silver layer
(
    df_Product_Categories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverProduct_Categories.delta")
    .save()
)

### Load the transformed Product_Categories table to gold layer
(
    df_Product_Categories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldProduct_Categories.delta")
    .save()
)
display(df_Product_Categories.head(10))

ProductCategoryKey,CategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories


#### Product_SubCategories Table (No transformation)

In [0]:

paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Product_Subcategories"

df_Product_SubCategories = (
    spark.read
    .format("parquet")
    .load(paths)
)

### Load the transformed Product_SubCategories table to silver layer
(
    df_Product_SubCategories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverProduct_Subcategories.delta")
    .save()
)

### Load the transformed Product_SubCategories table to gold layer
(
    df_Product_SubCategories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldProduct_Subcategories.delta")
    .save()
)
display(df_Product_SubCategories.head(5))

ProductSubcategoryKey,SubcategoryName,ProductCategoryKey
1,Mountain Bikes,1
2,Road Bikes,1
3,Touring Bikes,1
4,Handlebars,2
5,Bottom Brackets,2


#### Product Table

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Products"

df_Product = (
    spark.read
    .format("parquet")
    .load(paths)
)


df_Product = (
    df_Product
    .withColumn("ProductCost", F.round(F.col("ProductCost"), 2))
    .withColumn("ProductPrice", F.round(F.col("ProductPrice"), 2))
    .drop("ProductDescription", "ProductName")
)

### Load the transformed Product table to silver layer
(
    df_Product.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverProducts.delta")
    .save()
)

### Load the transformed Product table to Gold layer
(
    df_Product.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldProducts.delta")
    .save()
)
display(df_Product.head(5))

ProductKey,ProductSubcategoryKey,ProductSKU,ModelName,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL-U509-R,Sport-100,Red,0,0,13.09,34.99
215,31,HL-U509,Sport-100,Black,0,0,12.03,33.64
218,23,SO-B909-M,Mountain Bike Socks,White,M,U,3.4,9.5
219,23,SO-B909-L,Mountain Bike Socks,White,L,U,3.4,9.5
220,31,HL-U509-B,Sport-100,Blue,0,0,12.03,33.64


#### Sales Table (No Transformation)

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Sales"

df_Sales = (
    spark.read
    .format("parquet")
    .load(paths)
)

### Load the transformed Sales table
(
    df_Sales.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverSales.delta")
    .save()
)


### Load the transformed Sales table to gold layer to silver layer
(
    df_Sales.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldSales.delta")
    .save()
)
display(df_Sales.head(10))

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2015-01-01,2001-09-21,SO45080,332,14657,1,1,1
2015-01-01,2001-12-05,SO45079,312,29255,4,1,1
2015-01-01,2001-10-29,SO45082,350,11455,9,1,1
2015-01-01,2001-11-16,SO45081,338,26782,6,1,1
2015-01-02,2001-12-15,SO45083,312,14947,10,1,1
2015-01-02,2001-10-12,SO45084,310,29143,4,1,1
2015-01-02,2001-12-18,SO45086,314,18747,9,1,1
2015-01-02,2001-10-09,SO45085,312,18746,9,1,1
2015-01-03,2001-10-03,SO45093,312,18906,9,1,1
2015-01-03,2001-09-29,SO45090,310,29170,4,1,1


#### Retuns Table (No Transformation)

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Returns"

df_Returns = (
    spark.read
    .format("parquet")
    .load(paths)
)


### Load the transformed Returns table to silver layer
(
    df_Returns.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverReturns.delta")
    .save()
)

### Load the transformed Returns table to gold layer
(
    df_Returns.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldReturns.delta")
    .save()
)
display(df_Returns.head(5))

ReturnDate,TerritoryKey,ProductKey,ReturnQuantity
2015-01-18,9,312,1
2015-01-18,10,310,1
2015-01-21,8,346,1
2015-01-22,4,311,1
2015-02-02,6,312,1


#### Territories Table (No Transformation)

In [0]:
paths = "abfss://demo@jerryadls.dfs.core.windows.net/bronze/AdventureWorks_Territories"

df_Territories = (
    spark.read
    .format("parquet")
    .load(paths)
)

### Load the transformed Territories table to silver layer
(
    df_Territories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/silver/SilverTerritories.delta")
    .save()
)

### Load the transformed Territories table to gold layer
(
    df_Territories.write
    .format("delta")
    .mode("append")
    .option("path", "abfss://demo@jerryadls.dfs.core.windows.net/gold/GoldTerritories.delta")
    .save()
)
display(df_Territories.head(5))


SalesTerritoryKey,Region,Country,Continent
1,Northwest,United States,North America
2,Northeast,United States,North America
3,Central,United States,North America
4,Southwest,United States,North America
5,Southeast,United States,North America
